# Acoustic Fault Diagnosis Baseline — HUSTmotor

This notebook was developed as part of a **workshop project at the Polytechnic University of Tirana (UPT)** on multimodal deep learning for electrical motor fault diagnosis using synchronized **sound and vibration signals**.

My contribution focuses on the **acoustic modality**. The goal is to build and evaluate an acoustic-only **1D CNN baseline** and study its ability to generalize across different motor operating speeds using a leave-one-speed-out protocol.

The implementation follows the CNN architecture and experimental setting described in:

> C. Zhao, W. Shen, E. Zio, and H. Ma, *“Multimodal unified generalization and translation network for intelligent fault diagnosis under dynamic environments,”* Engineering Applications of Artificial Intelligence, vol. 162, 2025, 112559.  
> DOI: https://doi.org/10.1016/j.engappai.2025.112559

Original dataset:

> [HUSTmotor Multi-Modal Dataset](https://github.com/CHAOZHAO-1/HUSTmotor-multi-modal-dataset)

Shared preprocessing used by this notebook:

> [Gentian Qinami — ISCF Project Preprocessing](https://github.com/Genti24-bit/ISCF_Project/)

The notebook covers dataset setup, domain-generalization task loading, the paper-inspired 1D CNN, training, evaluation, ten-run aggregation, and result visualization.

> **Recommended environment:** Google Colab with GPU acceleration.


In [ ]:
import sys, platform, torch
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: enable a GPU in Colab.")

## 1. Mount Drive and create all folders automatically

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/HUSTmotor_Acoustic_Baseline")

DATA_DIR = PROJECT_DIR / "data"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = PROJECT_DIR / "figures"

for p in [DATA_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)

## 2. Clone Genti's repository

In [ ]:
import subprocess, shutil
REPO_URL = "https://github.com/Genti24-bit/ISCF_Project.git"
REPO_DIR = Path("/content/ISCF_Project")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

print("Repo:", REPO_DIR)
print("Commit:", commit)

## 3. Download and verify `hustmotor_1024.npz`

Expected SHA-256:

`bd4e3c7273fa5efced76a39863e7a7f80255b8e382383519d613492cf671c658`

The file is stored in Drive so it persists across Colab sessions.

In [ ]:
import hashlib, requests
from tqdm.auto import tqdm

DATA_PATH = DATA_DIR / "hustmotor_1024.npz"
EXPECTED_SHA256 = "bd4e3c7273fa5efced76a39863e7a7f80255b8e382383519d613492cf671c658"

def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_data(path):
    api = "https://api.github.com/repos/Genti24-bit/ISCF_Project/releases/tags/v1.0"
    release = requests.get(api, timeout=30)
    release.raise_for_status()
    assets = release.json()["assets"]
    asset = next(a for a in assets if a["name"] == "hustmotor_1024.npz")
    url = asset["browser_download_url"]

    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in r.iter_content(1024*1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

if not DATA_PATH.exists() or file_hash(DATA_PATH) != EXPECTED_SHA256:
    if DATA_PATH.exists():
        DATA_PATH.unlink()
    print("Downloading data v1.0...")
    download_data(DATA_PATH)

actual = file_hash(DATA_PATH)
print("Dataset:", DATA_PATH)
print("SHA-256:", actual)
assert actual == EXPECTED_SHA256, "Wrong dataset hash. Do not train."
print("Hash check PASSED.")

## 4. Import the shared loader and inspect the dataset

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt

sys.path.insert(0, str(REPO_DIR))
from preprocess_hustmotor import load_dg_task

d = np.load(DATA_PATH)

for key in d.files:
    print(f"{key:10s} {d[key].shape} {d[key].dtype}")

print("Classes:", d["classes"])
print("Domains:", d["domains"])
print("Meta:", d["meta"])

## 5. Load Task 9 first

Task 9 trains on 10/20/30 Hz and tests on unseen 5 Hz.

The acoustic arrays are `Xa_tr`, `Xa_va`, `Xa_te`.  
**Do not normalize them again**: data v1.0 already uses per-window z-score normalization.

In [ ]:
TASK_TO_DOMAIN = {9:0, 10:1, 11:2, 12:3}
DOMAIN_TO_SPEED = {0:"5 Hz", 1:"10 Hz", 2:"20 Hz", 3:"30 Hz"}
CLASS_NAMES = [
    "Healthy", "Bearing fault", "Bowed rotor",
    "Broken rotor bars", "Rotor misalignment", "Voltage unbalance"
]

TASK = 9
t = load_dg_task(str(DATA_PATH), target_domain=TASK_TO_DOMAIN[TASK])

Xv_tr, Xa_tr, y_tr = t["train"]
Xv_va, Xa_va, y_va = t["val"]
Xv_te, Xa_te, y_te = t["target"]

print("Train:", Xa_tr.shape, y_tr.shape)
print("Val:  ", Xa_va.shape, y_va.shape)
print("Test: ", Xa_te.shape, y_te.shape)

assert Xa_tr.shape == (14400, 1024)
assert Xa_va.shape == (3600, 1024)
assert Xa_te.shape == (1200, 1024)

print("\nMean of per-window means:", Xa_tr.mean(axis=1).mean())
print("Mean of per-window stds :", Xa_tr.std(axis=1).mean())
print("\nDo NOT normalize again.")

## 6. Inspect class balance and example acoustic windows

In [ ]:
def counts(y):
    return pd.DataFrame({
        "class": CLASS_NAMES,
        "count": np.bincount(y.astype(np.int64), minlength=6)
    })

display(counts(y_tr))
display(counts(y_va))
display(counts(y_te))

FS = 25_600
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for c, ax in enumerate(axes.ravel()):
    i = np.where(y_tr == c)[0][0]
    x = Xa_tr[i]
    time = np.arange(1024) / FS
    ax.plot(time, x)
    ax.set_title(CLASS_NAMES[c])
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Normalized amplitude")
    ax.grid(alpha=.25)
plt.tight_layout()
plt.show()

## 7. PyTorch DataLoaders

`Conv1d` expects `(batch, channels, length)`, so `(N,1024)` becomes `(N,1,1024)`.

In [ ]:
import random
from torch.utils.data import TensorDataset, DataLoader

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 256

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_loader(X, y, shuffle):
    X = torch.from_numpy(X).float().unsqueeze(1)
    y = torch.from_numpy(y.astype(np.int64)).long()
    return DataLoader(
        TensorDataset(X, y),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=(DEVICE.type == "cuda")
    )

train_loader = make_loader(Xa_tr, y_tr, True)
val_loader = make_loader(Xa_va, y_va, False)
test_loader = make_loader(Xa_te, y_te, False)

xb, yb = next(iter(train_loader))
print("Device:", DEVICE)
print("X batch:", xb.shape)
print("y batch:", yb.shape, yb.dtype)

## 8. Paper-style acoustic 1D CNN

Feature extractor from Table 1:
- Conv1D 16, kernel 64 -> pool 2
- Conv1D 32, kernel 16 -> pool 2
- Conv1D 64, kernel 5 -> pool 2
- Conv1D 64, kernel 5 -> pool 2
- fully connected classifier -> 6 classes

Implementation assumptions because the paper does not specify them:
- `padding=0` (valid convolution)
- `Conv -> BatchNorm -> ReLU -> MaxPool`

We do **not** put Softmax inside `forward()` because `CrossEntropyLoss` expects logits.

In [ ]:
import torch.nn as nn

class AcousticCNN1D(nn.Module):
    def __init__(self, n_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 16, 64, stride=1, padding=0),
            nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2,2),

            nn.Conv1d(16, 32, 16, stride=1, padding=0),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2,2),

            nn.Conv1d(32, 64, 5, stride=1, padding=0),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2,2),

            nn.Conv1d(64, 64, 5, stride=1, padding=0),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2,2),
        )

        with torch.no_grad():
            z = self.features(torch.zeros(1,1,1024))
            self.feature_shape = tuple(z.shape[1:])
            flat_dim = z.flatten(1).shape[1]

        self.classifier = nn.Linear(flat_dim, n_classes)

    def extract_features(self, x):
        return self.features(x).flatten(1)

    def forward(self, x):
        return self.classifier(self.extract_features(x))

model = AcousticCNN1D().to(DEVICE)
print(model)
print("Feature shape:", model.feature_shape)

with torch.no_grad():
    out = model(torch.zeros(8,1,1024).to(DEVICE))
print("Output:", out.shape)
assert out.shape == (8,6)

## 9. Training utilities

Paper settings:
- Adam
- 50 epochs
- batch size 256
- `lr = 0.0005 / (1 + 10*p)^0.75`
- `p = epoch / total_epochs`

Model selection uses the **source validation set only**.  
The unseen target is evaluated only after the best validation checkpoint has been chosen.

In [ ]:
from copy import deepcopy

BASE_LR = 0.0005
EPOCHS = 50

def paper_lr(epoch, total_epochs):
    p = epoch / total_epochs
    return BASE_LR / ((1 + 10*p) ** 0.75)

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, n = 0.0, 0, 0
    ys, ps = [], []

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss = criterion(logits, y)
        pred = logits.argmax(1)

        loss_sum += loss.item() * y.size(0)
        correct += (pred == y).sum().item()
        n += y.size(0)
        ys.append(y.cpu())
        ps.append(pred.cpu())

    return {
        "loss": loss_sum/n,
        "accuracy": correct/n,
        "y_true": torch.cat(ys).numpy(),
        "y_pred": torch.cat(ps).numpy()
    }

def train_model(train_loader, val_loader, seed=0, epochs=50, verbose=True):
    set_seed(seed)
    model = AcousticCNN1D().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=BASE_LR)

    best_acc, best_epoch, best_state = -1, -1, None
    hist = []

    for epoch in range(epochs):
        lr = paper_lr(epoch, epochs)
        for g in optimizer.param_groups:
            g["lr"] = lr

        model.train()
        loss_sum, correct, n = 0.0, 0, 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            loss_sum += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += y.size(0)

        train_loss = loss_sum/n
        train_acc = correct/n
        val = evaluate(model, val_loader, criterion)

        hist.append({
            "epoch": epoch+1,
            "lr": lr,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val["loss"],
            "val_acc": val["accuracy"]
        })

        if val["accuracy"] > best_acc:
            best_acc = val["accuracy"]
            best_epoch = epoch+1
            best_state = deepcopy(model.state_dict())

        if verbose and (epoch == 0 or (epoch+1)%5 == 0 or epoch+1 == epochs):
            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"train={train_acc:.4f} | val={val['accuracy']:.4f} | lr={lr:.6f}"
            )

    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), best_epoch, best_acc

## 10. Smoke test (2 epochs only)

In [ ]:
smoke_model, smoke_hist, _, _ = train_model(
    train_loader, val_loader, seed=0, epochs=2, verbose=True
)
display(smoke_hist)
print("Smoke test finished.")

## 11. Real Task 9 experiment (50 epochs)

After training is finished and the best source-validation checkpoint is selected, we evaluate the unseen 5 Hz target **once**.

In [ ]:
TASK = 9
SEED = 0

model, history, best_epoch, best_val = train_model(
    train_loader, val_loader, seed=SEED, epochs=50, verbose=True
)

target = evaluate(model, test_loader, nn.CrossEntropyLoss())

print("\nTask 9")
print("Best source-val epoch:", best_epoch)
print(f"Best val accuracy: {100*best_val:.2f}%")
print(f"Unseen 5 Hz target accuracy: {100*target['accuracy']:.2f}%")

model_path = MODELS_DIR / f"acoustic_task{TASK}_seed{SEED}.pt"
torch.save({
    "task": TASK,
    "seed": SEED,
    "best_epoch": best_epoch,
    "best_val_acc": best_val,
    "target_acc": target["accuracy"],
    "model_state_dict": model.state_dict(),
    "dataset_sha256": EXPECTED_SHA256,
    "git_commit": commit
}, model_path)

history.to_csv(
    RESULTS_DIR / f"history_task{TASK}_seed{SEED}.csv",
    index=False
)

print("Saved:", model_path)

## 12. Learning curve

In [ ]:
plt.figure(figsize=(8,4.5))
plt.plot(history["epoch"], 100*history["train_acc"], label="Train")
plt.plot(history["epoch"], 100*history["val_acc"], label="Validation")
plt.axvline(best_epoch, linestyle="--", alpha=.6, label=f"Best epoch {best_epoch}")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Task 9 — Acoustic CNN")
plt.grid(alpha=.25)
plt.legend()
plt.tight_layout()

p = FIGURES_DIR / "task9_learning_curve.png"
plt.savefig(p, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", p)

## 13. Confusion matrix and class-wise metrics

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

y_true, y_pred = target["y_true"], target["y_pred"]
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, xticks_rotation=35, values_format="d", colorbar=False
)
ax.set_title("Task 9 — Acoustic-only CNN — unseen 5 Hz")
plt.tight_layout()

p = FIGURES_DIR / "task9_confusion_matrix.png"
plt.savefig(p, dpi=200, bbox_inches="tight")
plt.show()

print(classification_report(
    y_true, y_pred, target_names=CLASS_NAMES, digits=4
))

## 14. Function to run any of Tasks 9–12

Paper acoustic-only reference:
- Task 9: 49.23%
- Task 10: 61.89%
- Task 11: 59.14%
- Task 12: 28.13%
- Average: 49.60%

Your numbers do not have to match exactly because data v1.0 uses a block-based split to avoid leakage, whereas the paper describes a random 800/200 split.

In [ ]:
PAPER_ACOUSTIC = {9:49.23, 10:61.89, 11:59.14, 12:28.13}

def run_task(task_number, seed=0, epochs=50, verbose=False):
    td = TASK_TO_DOMAIN[task_number]
    t = load_dg_task(str(DATA_PATH), target_domain=td)

    _, Xtr, ytr = t["train"]
    _, Xva, yva = t["val"]
    _, Xte, yte = t["target"]

    tr = make_loader(Xtr, ytr, True)
    va = make_loader(Xva, yva, False)
    te = make_loader(Xte, yte, False)

    model, hist, best_epoch, best_val = train_model(
        tr, va, seed=seed, epochs=epochs, verbose=verbose
    )

    # Touch unseen target only now.
    target = evaluate(model, te, nn.CrossEntropyLoss())

    torch.save(
        {
            "task": task_number,
            "seed": seed,
            "best_epoch": best_epoch,
            "best_val_acc": best_val,
            "target_acc": target["accuracy"],
            "model_state_dict": model.state_dict()
        },
        MODELS_DIR / f"acoustic_task{task_number}_seed{seed}.pt"
    )

    return {
        "task": task_number,
        "target_speed": DOMAIN_TO_SPEED[td],
        "seed": seed,
        "best_epoch": best_epoch,
        "val_acc": 100*best_val,
        "target_acc": 100*target["accuracy"],
        "paper_acoustic_acc": PAPER_ACOUSTIC[task_number]
    }

## 15. Run Tasks 9–12 for one seed

In [ ]:
results = []

for task_number in [9,10,11,12]:
    print(f"\n--- Task {task_number} ---")
    r = run_task(task_number, seed=0, epochs=50, verbose=False)
    results.append(r)
    print(
        f"Our target: {r['target_acc']:.2f}% | "
        f"Paper M2: {r['paper_acoustic_acc']:.2f}%"
    )

results_df = pd.DataFrame(results)
display(results_df)

print("Our average:", results_df["target_acc"].mean())
print("Paper M2 average: 49.60%")

results_df.to_csv(
    RESULTS_DIR / "acoustic_tasks_9_12_seed0.csv",
    index=False
)

## 16. Final paper-style experiment: 10 independent trials

The paper averages 10 runs. This means **40 trainings** (4 tasks × 10 seeds), so do this only after everything above works.

Change `RUN_10_TRIALS` to `True` when ready.

In [ ]:
RUN_10_TRIALS = False

if RUN_10_TRIALS:
    rows = []

    for seed in range(10):
        for task_number in [9,10,11,12]:
            print(f"Seed {seed} | Task {task_number}")
            rows.append(
                run_task(task_number, seed=seed, epochs=50, verbose=False)
            )

    raw_df = pd.DataFrame(rows)
    raw_df.to_csv(
        RESULTS_DIR / "acoustic_10runs_raw.csv", index=False
    )

    summary = (
        raw_df.groupby(["task","target_speed"])
        .agg(
            mean_accuracy=("target_acc","mean"),
            std_accuracy=("target_acc","std"),
            mean_best_epoch=("best_epoch","mean")
        )
        .reset_index()
    )

    summary["paper_acoustic_acc"] = summary["task"].map(PAPER_ACOUSTIC)

    display(summary)
    print("Overall mean:", summary["mean_accuracy"].mean())

    summary.to_csv(
        RESULTS_DIR / "acoustic_10runs_summary.csv", index=False
    )
else:
    print("Disabled. Set RUN_10_TRIALS = True when ready.")

## Aggregate Evaluation Across 10 Independent Runs

After completing the 10 independent training runs for Tasks 9–12, this section re-evaluates the saved checkpoints to measure variability and class-wise behavior.

It computes:
- mean target accuracy and standard deviation;
- class-wise precision, recall and F1-score;
- mean normalized confusion matrices;
- comparison with the paper's acoustic-only M2 baseline.

All checkpoints are **reloaded for evaluation only; no retraining is performed here**.


In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score

TASKS = [9, 10, 11, 12]
SEEDS = list(range(10))

missing = []
for task in TASKS:
    for seed in SEEDS:
        p = MODELS_DIR / f"acoustic_task{task}_seed{seed}.pt"
        if not p.exists():
            missing.append(str(p))

if missing:
    raise FileNotFoundError(f"{len(missing)} checkpoint(s) missing. First run Block 16.")
print("All 40 checkpoints found.")

In [ ]:
class_metric_rows = []
task_accuracy_rows = []
confusion_matrices = {task: [] for task in TASKS}
criterion = nn.CrossEntropyLoss()

for task in TASKS:
    target_domain = TASK_TO_DOMAIN[task]
    task_data = load_dg_task(str(DATA_PATH), target_domain=target_domain)
    _, Xa_test, y_test = task_data["target"]
    test_loader_task = make_loader(Xa_test, y_test, shuffle=False)

    for seed in SEEDS:
        checkpoint = torch.load(
            MODELS_DIR / f"acoustic_task{task}_seed{seed}.pt",
            map_location=DEVICE
        )
        model = AcousticCNN1D().to(DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
        metrics = evaluate(model, test_loader_task, criterion)

        y_true, y_pred = metrics["y_true"], metrics["y_pred"]
        task_accuracy_rows.append({
            "task": task,
            "target_speed": DOMAIN_TO_SPEED[target_domain],
            "seed": seed,
            "target_accuracy": 100 * accuracy_score(y_true, y_pred)
        })

        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred,
            labels=np.arange(len(CLASS_NAMES)),
            zero_division=0
        )
        for class_id, class_name in enumerate(CLASS_NAMES):
            class_metric_rows.append({
                "task": task,
                "target_speed": DOMAIN_TO_SPEED[target_domain],
                "seed": seed,
                "class_id": class_id,
                "class_name": class_name,
                "precision": precision[class_id],
                "recall": recall[class_id],
                "f1": f1[class_id],
                "support": support[class_id]
            })

        confusion_matrices[task].append(
            confusion_matrix(
                y_true, y_pred,
                labels=np.arange(len(CLASS_NAMES)),
                normalize="true"
            )
        )

print("Evaluation of all 40 checkpoints completed.")

In [ ]:
class_metrics_df = pd.DataFrame(class_metric_rows)
task_accuracy_df = pd.DataFrame(task_accuracy_rows)

class_summary = (
    class_metrics_df
    .groupby(["task", "target_speed", "class_name"])
    .agg(
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std")
    )
    .reset_index()
)

metric_cols = [
    "precision_mean","precision_std",
    "recall_mean","recall_std",
    "f1_mean","f1_std"
]
class_summary[metric_cols] *= 100
display(class_summary)

class_summary.to_csv(
    RESULTS_DIR / "acoustic_class_metrics_10runs_summary.csv",
    index=False
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 13))
axes = axes.flatten()

for ax, task in zip(axes, TASKS):
    mean_cm = np.mean(confusion_matrices[task], axis=0) * 100
    ConfusionMatrixDisplay(
        mean_cm,
        display_labels=CLASS_NAMES
    ).plot(
        ax=ax,
        values_format=".1f",
        xticks_rotation=40,
        colorbar=False
    )
    ax.set_title(
        f"Task {task} — Unseen {DOMAIN_TO_SPEED[TASK_TO_DOMAIN[task]]}\n"
        "Mean over 10 runs (%)"
    )

plt.suptitle("Acoustic-only CNN — Mean Normalized Confusion Matrices", fontsize=16)
plt.tight_layout(rect=[0,0,1,0.96])
plt.savefig(
    FIGURES_DIR / "acoustic_confusion_matrices_10runs.png",
    dpi=250,
    bbox_inches="tight"
)
plt.show()

In [ ]:
task_summary = (
    task_accuracy_df
    .groupby(["task", "target_speed"])
    .agg(
        mean_accuracy=("target_accuracy", "mean"),
        std_accuracy=("target_accuracy", "std"),
        min_accuracy=("target_accuracy", "min"),
        max_accuracy=("target_accuracy", "max")
    )
    .reset_index()
)

task_summary["paper_M2"] = task_summary["task"].map(PAPER_ACOUSTIC)
task_summary["difference_pp"] = (
    task_summary["mean_accuracy"] - task_summary["paper_M2"]
)

display(task_summary)
task_summary.to_csv(
    RESULTS_DIR / "acoustic_final_comparison_10runs.csv",
    index=False
)

In [ ]:
tasks = task_summary["task"].to_numpy()
ours_mean = task_summary["mean_accuracy"].to_numpy()
ours_std = task_summary["std_accuracy"].to_numpy()
paper = task_summary["paper_M2"].to_numpy()

x = np.arange(len(tasks))
width = 0.36
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.bar(x - width/2, paper, width, label="Reference paper M2")
ax.bar(
    x + width/2,
    ours_mean,
    width,
    yerr=ours_std,
    capsize=5,
    label="Our acoustic CNN (10 runs)"
)
ax.set_xticks(x)
ax.set_xticklabels([
    f"Task {task}\nTarget {DOMAIN_TO_SPEED[TASK_TO_DOMAIN[task]]}"
    for task in tasks
])
ax.set_ylabel("Target accuracy (%)")
ax.set_title("Acoustic-only Domain Generalization:\nOur Implementation vs Reference Paper")
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "acoustic_paper_vs_ours_10runs.png",
    dpi=250,
    bbox_inches="tight"
)
plt.show()

In [ ]:
seed_overall = task_accuracy_df.groupby("seed")["target_accuracy"].mean()
overall_mean = seed_overall.mean()
overall_std = seed_overall.std()

print(
    f"Overall acoustic accuracy across Tasks 9–12: "
    f"{overall_mean:.2f}% ± {overall_std:.2f}%"
)
print("Reference paper M2 average: 49.60%")
print(f"Difference: {overall_mean - 49.60:+.2f} percentage points")

In [ ]:
task_macro_summary = (
    class_metrics_df
    .groupby(["task", "target_speed", "seed"])
    .agg(
        macro_precision=("precision", "mean"),
        macro_recall=("recall", "mean"),
        macro_f1=("f1", "mean")
    )
    .reset_index()
    .groupby(["task", "target_speed"])
    .agg(
        macro_precision_mean=("macro_precision", "mean"),
        macro_precision_std=("macro_precision", "std"),
        macro_recall_mean=("macro_recall", "mean"),
        macro_recall_std=("macro_recall", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std")
    )
    .reset_index()
)

for col in task_macro_summary.columns:
    if col not in ["task", "target_speed"]:
        task_macro_summary[col] *= 100

display(task_macro_summary)
task_macro_summary.to_csv(
    RESULTS_DIR / "acoustic_task_macro_metrics_10runs.csv",
    index=False
)

# Expected output in Drive

```text
HUSTmotor_Acoustic_Baseline/
├── data/
│   └── hustmotor_1024.npz
├── models/
│   └── acoustic_task*_seed*.pt
├── results/
│   ├── history_task9_seed0.csv
│   ├── acoustic_tasks_9_12_seed0.csv
│   ├── acoustic_10runs_raw.csv
│   ├── acoustic_10runs_summary.csv
│   ├── acoustic_class_metrics_10runs_summary.csv
│   ├── acoustic_final_comparison_10runs.csv
│   └── acoustic_task_macro_metrics_10runs.csv
└── figures/
    ├── task9_learning_curve.png
    ├── task9_confusion_matrix.png
    ├── acoustic_confusion_matrices_10runs.png
    └── acoustic_paper_vs_ours_10runs.png
```
